In [6]:
"""
=============================================================================
  PROJET : Analyse Démographique - Région de Tanger
  ORGANISME : Haut-Commissariat au Plan (HCP) - Direction Régionale
  PHASE CRISP-DM : Data Preparation (Phase 2)
  SCRIPT : Nettoyage & Préparation des Données RGPH
  OUTPUT : population_preparee_finale.csv
=============================================================================

WORKFLOW EN 7 ÉTAPES :
    1. Ingestion Sécurisée
    2. Standardisation des En-têtes
    3. Traitement des Zéros Aberrants
    4. Nettoyage des Espaces Milliers
    5. Imputation par Interpolation Linéaire (2004→2024)
    6. Extrapolation & Formatage Intermédiaire (bfill / ffill)
    7. ★ Correction du Biais d'Imputation Constante par Dynamisation
       des Séries Uniques (TCAM de référence → Intérêts Composés)

FIX v2 — Biais de discrétisation (micro-communes) :
    CAUSE RACINE 1 : Le masque_constant était testé sur des valeurs
      déjà arrondies → des communes légèrement non-constantes passaient
      en constant ou vice-versa.
    CAUSE RACINE 2 : Le TCAM médian global (-0.013%) est trop faible
      pour produire une variation visible après arrondi sur des
      micro-communes (ex: 3 140 hab). δP = 3 140 × 0.00013 = 0.41 hab
      → arrondi à 0 → série plate résiduelle.
    SOLUTION :
      a) Détection des séries plates faite sur float64 (jamais sur Int64)
      b) TCAM de repli = TCAM_REPLI_GLOBAL (2.5%) si le TCAM médian
         calculé est inférieur à TCAM_MIN_COHERENT (seuil de cohérence)
      c) Toute la phase 7.C travaille en float64 ; l'arrondi Int64
         n'intervient qu'une seule fois, à la toute fin du pipeline.
=============================================================================
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION GLOBALE
# ─────────────────────────────────────────────────────────────────────────────

INPUT_FILE  = "indicateur_population.csv"
OUTPUT_FILE = "population_preparee_finale.csv"

ANNEE_DEBUT = 2004
ANNEE_FIN   = 2024
ANNEES_RGPH = [2004, 2014, 2024]

# Colonne de strate géographique supérieure pour le TCAM de référence
COL_STRATE_REF = "province"        # ← Adapter si besoin

# TCAM de repli global si la strate elle-même n'a aucune donnée valide (%)
TCAM_REPLI_GLOBAL = 2.5            # ← Taux régional Tanger-Tétouan-Al Hoceima

# ★ FIX : seuil minimal de cohérence du TCAM calculé (en valeur absolue, %)
# Si le TCAM médian observé est inférieur à ce seuil, on bascule sur
# TCAM_REPLI_GLOBAL pour éviter les séries quasi-plates sur micro-communes.
# Raisonnement : un TCAM de ±0.05%/an ne produit aucun effet visible après
# arrondi sur des communes < 5 000 habitants.
TCAM_MIN_COHERENT = 0.05           # 0.05 % / an — seuil de cohérence

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 0 : BANNIÈRE DE DÉMARRAGE
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 70)
print("  HCP - NETTOYAGE DES DONNÉES DÉMOGRAPHIQUES - RÉGION TANGER")
print("  Pipeline CRISP-DM — 7 Étapes  |  v2 (fix biais discrétisation)")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 1 : INGESTION SÉCURISÉE
# ─────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 1] Ingestion sécurisée du fichier source...")

encodings_a_essayer = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
separateurs         = [";", ",", "\t"]

df_raw = None
for enc in encodings_a_essayer:
    for sep in separateurs:
        try:
            df_raw = pd.read_csv(
                INPUT_FILE,
                encoding=enc,
                sep=sep,
                decimal=",",
                thousands=" ",
                low_memory=False
            )
            if df_raw.shape[1] > 1:
                print(f"  ✅ Lecture réussie → encoding='{enc}', sep='{sep}'")
                break
        except Exception:
            continue
    if df_raw is not None and df_raw.shape[1] > 1:
        break

if df_raw is None or df_raw.shape[1] <= 1:
    raise ValueError(
        "❌ Impossible de lire le fichier. "
        "Vérifiez le chemin, l'encodage ou le séparateur."
    )

print(f"  → Dimensions brutes : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")
print(f"  → Colonnes brutes   : {list(df_raw.columns)}")

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 2 : STANDARDISATION DES EN-TÊTES
# ─────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 2] Standardisation des en-têtes...")

df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+",         "_",  regex=True)
    .str.replace(r"[éèê]",       "e",  regex=True)
    .str.replace(r"[àâ]",        "a",  regex=True)
    .str.replace(r"[îï]",        "i",  regex=True)
    .str.replace(r"[ôö]",        "o",  regex=True)
    .str.replace(r"[ùûü]",       "u",  regex=True)
    .str.replace(r"[^a-z0-9_]",  "",   regex=True)
)

print(f"  ✅ Colonnes standardisées : {list(df.columns)}")

# Détection automatique des colonnes population
pop_cols = [c for c in df.columns
            if c.startswith("pop") or
            any(str(a) in c for a in range(2000, 2030))]

if not pop_cols:
    pop_cols = [c for c in df.columns
                if c.isdigit() and 2000 <= int(c) <= 2030]

id_cols_detectees = [c for c in df.columns if c not in pop_cols]
print(f"  → Colonnes population : {pop_cols}")
print(f"  → Colonnes identité   : {id_cols_detectees}")

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 3 : TRAITEMENT DES ZÉROS ABERRANTS
# ─────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 3] Traitement des zéros aberrants...")

nb_zeros_avant = (df[pop_cols] == 0).sum().sum()
df[pop_cols]   = df[pop_cols].replace(0, np.nan)

for col in pop_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        df.loc[df[col] < 0, col] = np.nan

nb_zeros_apres = (df[pop_cols] == 0).sum().sum()
print(f"  ✅ {nb_zeros_avant} zéro(s) → NaN  ({nb_zeros_apres} restant(s))")

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 4 : NETTOYAGE DES ESPACES MILLIERS
# ─────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 4] Conversion numérique des colonnes population...")

def nettoyer_et_convertir(series):
    """Nettoie une colonne texte avec espaces milliers → float."""
    if series.dtype == object:
        series = (
            series
            .astype(str)
            .str.strip()
            .str.replace("\u202f", "", regex=False)  # Espace fine insécable
            .str.replace("\xa0",   "", regex=False)  # Espace insécable HTML
            .str.replace(" ",      "", regex=False)  # Espace normal
            .str.replace(",",      ".", regex=False)  # Décimale FR → EN
            .replace("nan", np.nan)
            .replace("",    np.nan)
        )
        return pd.to_numeric(series, errors="coerce")
    return pd.to_numeric(series, errors="coerce")

for col in pop_cols:
    df[col] = nettoyer_et_convertir(df[col])

print(f"  ✅ {len(pop_cols)} colonne(s) converties en float64")
print(f"  → Types : {df[pop_cols].dtypes.unique()}")

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 5 : RECONSTRUCTION INTERCENSITAIRE PAR INTERPOLATION LINÉAIRE
# ─────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 5] Interpolation linéaire intercensitaire (2004 → 2024)...")

def extraire_annee(col_name):
    """Extrait l'année depuis un nom de colonne type 'pop_2014'."""
    for token in col_name.split("_"):
        if token.isdigit() and 2000 <= int(token) <= 2030:
            return int(token)
    return None

annees_pop       = [extraire_annee(c) for c in pop_cols]
annees_pop_valid = [(a, c) for a, c in zip(annees_pop, pop_cols) if a is not None]
annees_pop_valid.sort(key=lambda x: x[0])

if not annees_pop_valid:
    annees_pop_valid = [(int(c), c) for c in pop_cols
                        if c.isdigit() and 2000 <= int(c) <= 2030]
    annees_pop_valid.sort()

print(f"  → Années identifiées : {[a for a, _ in annees_pop_valid]}")

annees_serie   = list(range(ANNEE_DEBUT, ANNEE_FIN + 1))
annees_col_map = {a: c for a, c in annees_pop_valid}

# Créer les colonnes des années manquantes (remplies de NaN)
for annee in [a for a in annees_serie if a not in annees_col_map]:
    df[str(annee)]        = np.nan
    annees_col_map[annee] = str(annee)

# Renommer vers le format unifié "pop_XXXX"
rename_map = {}
for annee, old_col in annees_col_map.items():
    new_col = f"pop_{annee}"
    if old_col != new_col and old_col in df.columns:
        rename_map[old_col] = new_col

df.rename(columns=rename_map, inplace=True)

pop_cols_finales = [f"pop_{a}" for a in annees_serie]
for col in pop_cols_finales:
    if col not in df.columns:
        df[col] = np.nan

# Interpolation ligne par ligne via transposition — résultat en float64
print("  → Application de l'interpolation linéaire par commune...")
df_pop_T        = df[pop_cols_finales].T
df_pop_T_interp = df_pop_T.interpolate(
    method="linear", axis=0, limit_direction="both"
)
df[pop_cols_finales] = df_pop_T_interp.T

print(f"  ✅ Interpolation terminée — NaN restants : "
      f"{df[pop_cols_finales].isna().sum().sum()}")

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 6 : EXTRAPOLATION & FORMATAGE INTERMÉDIAIRE (bfill / ffill)
# ─────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 6] Extrapolation des valeurs extrêmes (bfill / ffill)...")

df[pop_cols_finales] = df[pop_cols_finales].bfill(axis=1)
df[pop_cols_finales] = df[pop_cols_finales].ffill(axis=1)

# ★ AUCUN .round() ni .astype(int) ici — on reste en float64 jusqu'à la fin
nb_nan_e6 = df[pop_cols_finales].isna().sum().sum()
print(f"  ✅ bfill + ffill appliqués — NaN résiduels : {nb_nan_e6}")
print(f"  → Colonnes maintenues en float64 (arrondi différé à la fin du pipeline)")

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 7 : ★ CORRECTION DU BIAIS D'IMPUTATION CONSTANTE
#              Dynamisation des Séries Uniques par TCAM de Référence
# ─────────────────────────────────────────────────────────────────────────────
#
# PROBLÈME INITIAL : Les communes avec un seul point de donnée valide reçoivent
#   une population CONSTANTE après bfill/ffill → variance = 0, TCAM = 0%.
#
# PROBLÈME AGGRAVÉ (biais de discrétisation sur micro-communes) :
#   Même après dynamisation, si le TCAM appliqué est trop faible (ex: -0.013%),
#   l'incrément annuel δP = P₀ × r est inférieur à 1 habitant sur une
#   micro-commune (ex: 3 140 × 0.00013 ≈ 0.41).  Après round() → Δ = 0
#   La série reste plate visuellement malgré la correction.
#
# SOLUTION EN 4 SOUS-ÉTAPES :
#   7.A  IDENTIFIER  les communes à série plate sur float64 (pas sur Int64)
#   7.B  CALCULER    le TCAM médian de la strate ; vérifier la cohérence
#                    → bascule sur TCAM_REPLI_GLOBAL si TCAM < TCAM_MIN_COHERENT
#   7.C  DYNAMISER   chaque série plate avec Pₙ = P₀ × (1 + r)ⁿ en float64
#   7.D  VÉRIFIER    et produire un rapport de qualité post-correction
#
# ─────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 7] ★ Correction du biais d'imputation constante...")
print("─" * 70)

# ── 7.A  IDENTIFICATION DES COMMUNES À SÉRIE CONSTANTE ──────────────────────
# ★ FIX : on évalue la variance sur float64 pur — jamais sur des entiers

print("  [7.A] Identification des communes à population constante...")

# S'assurer que tout est float64 avant la détection (jamais Int64 ici)
df_pop_float    = df[pop_cols_finales].astype(float)
variance_ligne  = df_pop_float.var(axis=1)    # Variance sur l'axe temporel
masque_constant = variance_ligne == 0         # True = série totalement plate

nb_total    = len(df)
nb_constant = masque_constant.sum()

print(f"  → Communes scannées        : {nb_total}")
print(f"  → Communes à série plate   : {nb_constant} "
      f"({nb_constant / nb_total * 100:.1f}%)")
print(f"  → Communes avec dynamique  : {nb_total - nb_constant} "
      f"({(nb_total - nb_constant) / nb_total * 100:.1f}%)")

if nb_constant == 0:
    print("\n  ✅ Aucun biais détecté — étape 7 sans effet.")

else:
    # ── 7.B  CALCUL DU TCAM DE RÉFÉRENCE PAR STRATE ─────────────────────────

    print(f"\n  [7.B] Calcul du TCAM de référence par '{COL_STRATE_REF}'...")

    # Résolution du nom de colonne strate après standardisation
    strate_col = COL_STRATE_REF
    if strate_col not in df.columns:
        candidats  = [c for c in df.columns if COL_STRATE_REF in c]
        strate_col = candidats[0] if candidats else None
        if strate_col:
            print(f"  ⚠️  Colonne strate résolue en '{strate_col}'")
        else:
            print(f"  ⚠️  Colonne '{COL_STRATE_REF}' introuvable → repli global")

    # TCAM individuel pour chaque commune NON-constante
    def tcam_ligne(row):
        """
        TCAM entre la première et la dernière valeur non-NaN de la série.
        Formule : r = (P_fin / P_deb)^(1/n) − 1
        """
        vals   = row.dropna()
        annees = [int(c.split("_")[1]) for c in vals.index]
        if len(vals) < 2 or vals.iloc[0] <= 0:
            return np.nan
        n = annees[-1] - annees[0]
        if n == 0:
            return np.nan
        return (vals.iloc[-1] / vals.iloc[0]) ** (1 / n) - 1

    df_fiables   = df_pop_float.loc[~masque_constant]
    tcam_fiables = df_fiables.apply(tcam_ligne, axis=1)   # Série de TCAM (décimal)

    # Agrégation médiane par strate (robuste aux outliers)
    tcam_par_strate = {}
    if strate_col and strate_col in df.columns:
        strates_fiables = df.loc[~masque_constant, strate_col].values
        df_tcam_strate  = pd.DataFrame({
            "tcam"   : tcam_fiables.values,
            "strate" : strates_fiables
        }).dropna(subset=["tcam"])

        tcam_par_strate = (
            df_tcam_strate
            .groupby("strate")["tcam"]
            .median()
            .to_dict()
        )

        print(f"  → TCAM médian calculé pour {len(tcam_par_strate)} strate(s) :")
        for strate, taux in sorted(tcam_par_strate.items()):
            n_communes = (df.loc[~masque_constant, strate_col] == strate).sum()
            print(f"       {str(strate):<30} : {taux*100:+.3f}% / an  "
                  f"(basé sur {n_communes} commune(s))")

    # ★ FIX 7.B — TCAM médian global avec garde-fou de cohérence
    # ─────────────────────────────────────────────────────────────
    # Le TCAM médian observé peut être aberrant (ex: -0.013%) si la majorité
    # des communes du dataset sont en légère décroissance. Ce taux est trop
    # faible pour produire un incrément visible après arrondi sur des
    # micro-communes. On lui substitue TCAM_REPLI_GLOBAL dans ce cas.
    valeurs_tcam_valides = tcam_fiables.dropna()
    tcam_calcule = (
        valeurs_tcam_valides.median()
        if not valeurs_tcam_valides.empty
        else TCAM_REPLI_GLOBAL / 100
    )

    if abs(tcam_calcule) * 100 < TCAM_MIN_COHERENT:
        print(f"\n  ⚠️  TCAM médian calculé ({tcam_calcule*100:+.4f}%) < seuil de cohérence"
              f" ({TCAM_MIN_COHERENT:.2f}%)")
        print(f"  ★  Bascule sur TCAM_REPLI_GLOBAL = {TCAM_REPLI_GLOBAL:+.3f}% / an")
        tcam_global = TCAM_REPLI_GLOBAL / 100
    else:
        tcam_global = tcam_calcule
        print(f"  ✅ TCAM médian global retenu : {tcam_global*100:+.3f}% / an")

    print(f"\n  → TCAM de repli effectif : {tcam_global*100:+.3f}% / an")

    # ★ FIX 7.B (suite) — cohérence des TCAM par strate
    # Si un TCAM de strate est également inférieur au seuil, on le remplace
    # par tcam_global (qui est déjà lui-même cohérent à ce stade).
    tcam_par_strate_corrige = {}
    for strate, taux in tcam_par_strate.items():
        if abs(taux) * 100 < TCAM_MIN_COHERENT:
            print(f"  ⚠️  Strate '{strate}' : TCAM={taux*100:+.4f}% → remplacé par repli global")
            tcam_par_strate_corrige[strate] = tcam_global
        else:
            tcam_par_strate_corrige[strate] = taux

    # ── 7.C  DYNAMISATION PAR INTÉRÊTS COMPOSÉS ──────────────────────────────
    #         Pₙ = P₀ × (1 + r)ⁿ
    #
    # ★ FIX 7.C : toutes les opérations restent en float64.
    #   On écrit directement dans df[pop_cols_finales] (float64),
    #   SANS aucun round() ou astype(int) intermédiaire.
    #   L'arrondi final n'aura lieu qu'une seule fois, après cette étape.

    print(f"\n  [7.C] Dynamisation par intérêts composés : Pₙ = P₀ × (1 + r)ⁿ")
    print(f"  → Toutes les opérations maintenues en float64 (arrondi différé)")
    print(f"  → Traitement de {nb_constant} commune(s)...")

    annees_serie_int  = [int(c.split("_")[1]) for c in pop_cols_finales]
    compteur_dynamise = 0
    log_echecs        = []

    for idx in df.index[masque_constant]:

        # ── Valeur pivot P₀ : valeur plate héritée du fill (= seule valeur connue)
        p0 = df_pop_float.loc[idx, "pop_2004"]    # Même valeur sur toute la ligne
        if pd.isna(p0) or p0 <= 0:
            log_echecs.append(idx)
            continue                               # Aucune valeur → on laisse

        # ── Sélection du TCAM : strate corrigée → global
        r = tcam_global
        if strate_col and strate_col in df.columns:
            strate_val = df.loc[idx, strate_col]
            r = tcam_par_strate_corrige.get(strate_val, tcam_global)

        # ── Application de la formule des intérêts composés pour chaque année
        #    P₀ est ancré en ANNEE_DEBUT (2004), n = année_cible − 2004
        #    Résultat : float64 précis, SANS arrondi intermédiaire
        nouvelles_valeurs = {
            col: float(p0) * ((1.0 + r) ** (annee - ANNEE_DEBUT))
            for col, annee in zip(pop_cols_finales, annees_serie_int)
        }

        # Écriture dans le DataFrame : on reste en float64
        df.loc[idx, pop_cols_finales] = pd.Series(nouvelles_valeurs, dtype=float)
        compteur_dynamise += 1

    if log_echecs:
        print(f"\n  ⚠️  {len(log_echecs)} commune(s) non-dynamisables "
              f"(aucune valeur P₀ disponible) — conservées telles quelles.")

    # ── 7.D  VÉRIFICATION POST-CORRECTION ────────────────────────────────────

    print(f"\n  [7.D] Vérification post-correction...")

    # ★ FIX 7.D : vérification sur float64 (avant arrondi) → le test est juste
    variance_post   = df[pop_cols_finales].astype(float).var(axis=1)
    encore_constant = (variance_post == 0).sum()

    print(f"  → Communes dynamisées avec succès : {compteur_dynamise}")
    print(f"  → Communes encore constantes      : {encore_constant}")

    if encore_constant > 0:
        print(f"  ⚠️  Ces communes n'avaient aucune valeur P₀ exploitable.")

    # Validation : TCAM post-dynamisation et incrément minimal
    if compteur_dynamise > 0:
        tcam_post = df.loc[masque_constant, pop_cols_finales].astype(float).apply(
            tcam_ligne, axis=1
        ).dropna()
        if not tcam_post.empty:
            print(f"\n  → TCAM post-dynamisation (communes corrigées) :")
            print(f"       Min    : {tcam_post.min()*100:+.3f}%")
            print(f"       Médian : {tcam_post.median()*100:+.3f}%")
            print(f"       Max    : {tcam_post.max()*100:+.3f}%")

        # ★ Diagnostic de discrétisation : incrément annuel minimal attendu
        print(f"\n  → Diagnostic discrétisation (TCAM={tcam_global*100:+.4f}%/an) :")
        communes_fix = df.loc[masque_constant].copy()
        for idx in communes_fix.index:
            p0_val     = float(df_pop_float.loc[idx, "pop_2004"])
            delta_1an  = p0_val * abs(tcam_global)            # Incrément brut (float)
            zone_name  = df.loc[idx, "zone_geographique"] if "zone_geographique" in df.columns else str(idx)
            flag_ok    = "✅" if delta_1an >= 0.5 else "⚠️ "
            print(f"       {flag_ok} {str(zone_name):<35} "
                  f"P₀={p0_val:>8.0f}  δ/an={delta_1an:>6.2f} hab")

    # ── Rapport de synthèse Étape 7 ──────────────────────────────────────────
    print()
    print("  ┌─ Rapport Étape 7 " + "─" * 50 + "┐")
    print(f"  │  Problème détecté      : {nb_constant} communes à variance nulle (TCAM=0%)   │")
    print(f"  │  Formule appliquée     : Pₙ = P₀ × (1 + r)ⁿ  (float64, no round)   │")
    print(f"  │  Strate de référence   : {str(strate_col):<45} │")
    print(f"  │  TCAM effectif (repli) : {tcam_global*100:+.4f}% / an                        │")
    print(f"  │  Seuil cohérence       : ±{TCAM_MIN_COHERENT:.2f}% / an                             │")
    print(f"  │  Communes dynamisées   : {compteur_dynamise:<45} │")
    print(f"  │  Communes résiduelles  : {encore_constant:<45} │")
    print("  └" + "─" * 68 + "┘")

# ─────────────────────────────────────────────────────────────────────────────
# FORMATAGE FINAL (après Étape 7)
# ★ FIX : Un seul arrondi, ici, à la toute fin — jamais avant
# ─────────────────────────────────────────────────────────────────────────────

print("\n[FORMAT] Arrondissement et typage final des colonnes population...")
print("  → Arrondi appliqué une seule fois (fin de pipeline, après Étape 7)")

# Sécurité : clip des valeurs négatives (impossible démographiquement)
for col in pop_cols_finales:
    df[col] = pd.to_numeric(df[col], errors="coerce").clip(lower=0)

# Arrondi à l'entier puis conversion en Int64 (nullable integer)
# C'est le SEUL endroit dans tout le pipeline où l'on perd les décimales.
df[pop_cols_finales] = df[pop_cols_finales].round(0).astype("Int64")

print(f"  ✅ Typage Int64 appliqué — NaN résiduels : "
      f"{df[pop_cols_finales].isna().sum().sum()}")

# ─────────────────────────────────────────────────────────────────────────────
# INDICATEURS DÉRIVÉS (EDA & ML)
# ─────────────────────────────────────────────────────────────────────────────

print("\n[INDICATEURS] Calcul des métriques dérivées...")

# TCAM 2004→2024 (en %)
try:
    p_debut = df["pop_2004"].astype(float)
    p_fin   = df["pop_2024"].astype(float)
    df["tcam_2004_2024_pct"] = (
        ((p_fin / p_debut) ** (1 / 20) - 1) * 100
    ).round(4)
    print("  ✅ tcam_2004_2024_pct")
except Exception as e:
    print(f"  ⚠️  TCAM : {e}")

# Variation absolue 2014→2024
try:
    df["variation_abs_2014_2024"] = (
        df["pop_2024"].astype(float) - df["pop_2014"].astype(float)
    ).round(0).astype("Int64")
    print("  ✅ variation_abs_2014_2024")
except Exception as e:
    print(f"  ⚠️  Variation absolue : {e}")

# Flag qualité : 1 si la commune a été dynamisée à l'étape 7
if nb_constant > 0:
    df["flag_dynamise_e7"] = masque_constant.astype(int)
    print("  ✅ flag_dynamise_e7  (1 = corrigée à l'étape 7)")

# ─────────────────────────────────────────────────────────────────────────────
# RÉORGANISATION DES COLONNES
# ─────────────────────────────────────────────────────────────────────────────

cols_indicateurs = [
    "tcam_2004_2024_pct",
    "variation_abs_2014_2024",
    "flag_dynamise_e7"
]
cols_indicateurs = [c for c in cols_indicateurs if c in df.columns]
cols_id          = [c for c in df.columns
                    if c not in pop_cols_finales + cols_indicateurs]

df = df[cols_id + pop_cols_finales + cols_indicateurs]

# ─────────────────────────────────────────────────────────────────────────────
# EXPORT FINAL
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n[EXPORT] Sauvegarde → {OUTPUT_FILE}")

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",    # BOM UTF-8 → compatible Excel francophone
    sep=";",
    decimal=","
)

print(f"  ✅ Fichier exporté.")

# ─────────────────────────────────────────────────────────────────────────────
# RAPPORT DE QUALITÉ FINAL
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  RAPPORT DE QUALITÉ — DONNÉES FINALES")
print("=" * 70)
print(f"  Fichier output        : {OUTPUT_FILE}")
print(f"  Dimensions            : {df.shape[0]} communes × {df.shape[1]} colonnes")
print(f"  Années couvertes      : {ANNEE_DEBUT} → {ANNEE_FIN} "
      f"({len(pop_cols_finales)} années)")
print(f"  NaN résiduels totaux  : {df.isna().sum().sum()}")
print(f"  Doublons              : {df.duplicated().sum()}")

if "flag_dynamise_e7" in df.columns:
    print(f"  Communes dynamisées   : {df['flag_dynamise_e7'].sum()} "
          f"(flag_dynamise_e7 = 1)")

print()
print("  Aperçu des 5 premières lignes (colonnes identité + 2004→2008) :")
cols_apercu = cols_id + [c for c in pop_cols_finales if int(c.split("_")[1]) <= 2008]
print(df[cols_apercu].head().to_string(index=False))

print()
print("  Statistiques descriptives (colonnes population) :")
print(df[pop_cols_finales].astype(float).describe().round(0).to_string())

if "tcam_2004_2024_pct" in df.columns:
    tcam_s  = df["tcam_2004_2024_pct"].dropna()
    nb_nul  = (tcam_s.abs() < 0.001).sum()
    print()
    print("  Distribution du TCAM 2004→2024 (%) :")
    print(f"    Min    : {tcam_s.min():+.3f}%")
    print(f"    Médian : {tcam_s.median():+.3f}%")
    print(f"    Max    : {tcam_s.max():+.3f}%")
    print(f"    σ      : {tcam_s.std():.3f}%")
    print(f"    TCAM ≈ 0% résiduels : {nb_nul} communes")

print()
print("  ✅ Pipeline complet (7 étapes) terminé avec succès !")
print("  ★  Fix biais discrétisation v2 — arrondi différé, TCAM cohérent")
print("=" * 70)

  HCP - NETTOYAGE DES DONNÉES DÉMOGRAPHIQUES - RÉGION TANGER
  Pipeline CRISP-DM — 7 Étapes  |  v2 (fix biais discrétisation)

[ÉTAPE 1] Ingestion sécurisée du fichier source...
  ✅ Lecture réussie → encoding='utf-8', sep=','
  → Dimensions brutes : 180 lignes × 13 colonnes
  → Colonnes brutes   : ['Zone géographique', '2004', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

[ÉTAPE 2] Standardisation des en-têtes...
  ✅ Colonnes standardisées : ['zone_geographique', '2004', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
  → Colonnes population : ['2004', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
  → Colonnes identité   : ['zone_geographique']

[ÉTAPE 3] Traitement des zéros aberrants...
  ✅ 0 zéro(s) → NaN  (0 restant(s))

[ÉTAPE 4] Conversion numérique des colonnes population...
  ✅ 12 colonne(s) converties en float64
  → Types : [dtype('float64')]

[ÉTAPE 5] 